# House Price Prediction — Train & Predict

Một notebook duy nhất: load dữ liệu từ Supabase → xem dữ liệu training → train model → đánh giá → dự đoán giá cho toàn bộ listings.

> **Kernel:** chọn `Python (HousePricePrediction .venv)` rồi chạy các cell từ trên xuống dưới.

## 0. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import pickle

sys.path.insert(0, str(Path.cwd().parent))
from pipeline.supabase_handler import fetch_csv_from_supabase

print("✅ Imports successful")

## Part 1 — Training

### 1.1 Load dữ liệu từ Supabase

In [ ]:
df = fetch_csv_from_supabase("Raw_Features")

# Convert price from VND to billion VND
df['price_billion_vnd'] = df['price_vnd'] / 1e9

print(f"✓ Loaded {len(df)} records, {len(df.columns)} columns")
print(f"Price range: {df['price_billion_vnd'].min():.2f} – {df['price_billion_vnd'].max():.2f} billion VND")

# Save a snapshot of the raw data
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
df.to_csv(data_dir / 'raw_data.csv', index=False)
print("✓ Saved snapshot: data/raw_data.csv")

df.head(10)

### 1.2 Features & target

In [ ]:
# Numeric features (geospatial + property)
NUMERIC_COLS = [
    'nearest_school_km', 'school_count_3km',
    'nearest_hospital_km', 'hospital_count_5km',
    'nearest_marketplace_km', 'marketplace_count_3km',
    'nearest_supermarket_km', 'supermarket_count_3km',
    'nearest_mall_km', 'mall_count_3km',
    'nearest_bus_stop_km', 'bus_stop_count_1km',
    'nearest_metro_km', 'metro_count_5km',
    'area_m2', 'distance_to_center_km',
    'num_floors', 'num_bedrooms', 'road_width_m', 'width_m', 'length_m',
    'locality_population_density'
]

# Binary features (True/False → 0/1)
BIN_COLS = [
    'dining_room_bin', 'kitchen_bin', 'terrace_bin',
    'car_parking_bin', 'owner_listing_bin'
]

# Text features → one-hot encode
CAT_COLS = ['property_type', 'legal_status', 'direction']

# Dropped: locality, region, locality_square (location identifiers),
# listing_type (single value), link/title/street/addresses (free text), lat/lon
TARGET_COL = 'price_billion_vnd'

print(f"{len(NUMERIC_COLS)} numeric + {len(BIN_COLS)} binary + {len(CAT_COLS)} categorical, target = {TARGET_COL}")

### 1.3 Khám phá dữ liệu training

Thống kê mô tả và tỉ lệ thiếu dữ liệu của các cột dùng để train.

In [ ]:
# Descriptive statistics of numeric features + target
df[NUMERIC_COLS + [TARGET_COL]].describe().round(2).T

In [ ]:
# Missing values per training column
cols = NUMERIC_COLS + BIN_COLS + CAT_COLS + [TARGET_COL]
missing = df[cols].isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_table = pd.DataFrame({'missing': missing, 'missing_%': missing_pct})
missing_table[missing_table['missing'] > 0].sort_values('missing', ascending=False)

In [ ]:
# Binary & categorical features overview
for col in BIN_COLS:
    counts = df[col].value_counts()
    print(f"{col:22s} True: {counts.get(True, 0):>5} | False: {counts.get(False, 0):>5}")

for col in CAT_COLS:
    print(f"\n{col}:")
    print(df[col].fillna('(missing)').value_counts().to_string())

In [ ]:
# Distribution of the target (house prices)
prices = df[TARGET_COL].dropna()
prices_plot = prices[prices < 100]  # same outlier cutoff as training

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(prices_plot, bins=60, color='#4C78A8', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Price (billion VND)')
ax.set_ylabel('Number of listings')
ax.set_title('Target distribution — house prices (< 100B VND)')
ax.grid(True, alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Median price: {prices.median():.2f} billion VND")

### 1.4 Chuẩn bị dữ liệu

Quy tắc xử lý missing:

- `nearest_*` → điền **max** của cột (thiếu nghĩa là không có tiện ích gần)
- `length_m` / `width_m` → thiếu 1 trong 2 thì suy ra từ `area_m2 / cạnh còn lại`; thiếu cả 2 thì điền median
- `num_*`, `road_width_m` → điền median
- Thiếu `locality_population_density` → **drop dòng**
- Bin → 0/1, cột chữ → one-hot, loại outlier giá > 100 tỷ

In [ ]:
NEAREST_COLS = [c for c in NUMERIC_COLS if c.startswith('nearest_')]

def build_features(frame):
    """Điền missing + encode — dùng chung cho training và inference."""
    X = frame[NUMERIC_COLS].copy()

    # nearest_*: missing → max của cột
    X[NEAREST_COLS] = X[NEAREST_COLS].fillna(X[NEAREST_COLS].max())

    # length/width: suy ra từ area nếu thiếu 1 trong 2
    X['length_m'] = X['length_m'].fillna(
        (X['area_m2'] / X['width_m']).replace([np.inf, -np.inf], np.nan))
    X['width_m'] = X['width_m'].fillna(
        (X['area_m2'] / X['length_m']).replace([np.inf, -np.inf], np.nan))

    # Còn lại (num_*, road_width_m, length/width thiếu cả 2, ...): median
    X = X.fillna(X.median())

    # Binary → 0/1, categorical → one-hot
    return pd.concat([
        X,
        frame[BIN_COLS].astype(int),
        pd.get_dummies(frame[CAT_COLS].fillna('unknown')).astype(int),
    ], axis=1)

df_clean = df.dropna(subset=[TARGET_COL]).copy()
print(f"After dropping NaN prices: {len(df_clean)} records")

# Drop rows thiếu locality_population_density
n_before = len(df_clean)
df_clean = df_clean.dropna(subset=['locality_population_density'])
print(f"Dropped {n_before - len(df_clean)} rows missing locality_population_density")

# Report missing per group (trước khi điền)
n_nearest = df_clean[NEAREST_COLS].isnull().sum().sum()
n_len = df_clean['length_m'].isnull().sum()
n_wid = df_clean['width_m'].isnull().sum()
n_both = (df_clean['length_m'].isnull() & df_clean['width_m'].isnull()).sum()
n_med = df_clean[['num_floors', 'num_bedrooms', 'road_width_m']].isnull().sum()
print(f"\nMissing → nearest_* (fill max): {n_nearest}")
print(f"Missing → length_m: {n_len}, width_m: {n_wid} (thiếu cả 2 → median: {n_both})")
print(f"Missing → median fill: {n_med.to_dict()}")

X = build_features(df_clean)
FEATURE_COLS = list(X.columns)
y = df_clean[TARGET_COL].copy()

# Remove outliers (price > 100B VND)
mask = y < 100
X, y = X[mask], y[mask]
print(f"\n✓ Ready for training: X={X.shape} ({len(FEATURE_COLS)} features), y={y.shape}")

# Save the model-ready dataset (features + target)
ready = X.copy()
ready[TARGET_COL] = y
ready.to_csv(data_dir / 'model_ready_data.csv', index=False)
print("✓ Saved: data/model_ready_data.csv")

# The actual training data fed to the models:
X.head(10)

### 1.5 Chia train / test (80–20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

### 1.6 Train models

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"Random Forest      R²: {rf_r2:.4f} | RMSE: {rf_rmse:.4f} | MAE: {rf_mae:.4f} (billion VND)")

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)
gb_r2 = r2_score(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_mae = mean_absolute_error(y_test, gb_pred)
print(f"Gradient Boosting  R²: {gb_r2:.4f} | RMSE: {gb_rmse:.4f} | MAE: {gb_mae:.4f} (billion VND)")

#### TabPFN (tuỳ chọn — chậm nếu không có GPU)

⚠️ TabPFN chạy **rất chậm trên CPU**: benchmark trên máy này là ~26 phút cho 200 dòng
→ đánh giá đủ 1.589 dòng test mất **~3,5 giờ** (GPU/Colab thì chỉ vài giây).

- Không muốn chờ → **bỏ qua cell dưới**, các cell sau vẫn chạy bình thường (so sánh chỉ gồm 2 model).
- Muốn thử nhanh → giảm `N_EVAL` xuống (ví dụ 200 dòng, ~26 phút).

In [ ]:
# TabPFN — foundation model cho tabular data (in-context learning, không cần tune)
import time
from tabpfn import TabPFNRegressor

N_EVAL = len(X_test)  # giảm xuống (vd 200) để chạy nhanh hơn trên CPU

t0 = time.time()
tab = TabPFNRegressor()
tab.fit(X_train, y_train)  # chỉ lưu training set làm context
tab_pred = tab.predict(X_test.iloc[:N_EVAL])

tab_r2 = r2_score(y_test.iloc[:N_EVAL], tab_pred)
tab_rmse = np.sqrt(mean_squared_error(y_test.iloc[:N_EVAL], tab_pred))
tab_mae = mean_absolute_error(y_test.iloc[:N_EVAL], tab_pred)
print(f"TabPFN ({N_EVAL} test rows)  R²: {tab_r2:.4f} | RMSE: {tab_rmse:.4f} | MAE: {tab_mae:.4f} (billion VND)")
print(f"(fit + predict: {time.time() - t0:.0f}s)")

### 1.7 So sánh & chọn model tốt nhất

In [ ]:
results = {
    'random_forest':     (rf,  rf_r2,  rf_rmse,  rf_mae),
    'gradient_boosting': (gb,  gb_r2,  gb_rmse,  gb_mae),
}
if 'tab' in globals():  # chỉ thêm nếu cell TabPFN đã chạy
    results['tabpfn'] = (tab, tab_r2, tab_rmse, tab_mae)

comparison = pd.DataFrame(
    [(name.replace('_', ' ').title(), r2, rmse, mae)
     for name, (_, r2, rmse, mae) in results.items()],
    columns=['Model', 'R²', 'RMSE (billion VND)', 'MAE (billion VND)'],
).round(4)

best_name = max(results, key=lambda k: results[k][1])
best_model, best_r2 = results[best_name][0], results[best_name][1]
print(f"✅ Best model: {best_name.upper()} (R² = {best_r2:.4f})")

comparison

### 1.8 Đánh giá model tốt nhất

In [ ]:
y_best_pred = best_model.predict(X_test)
errors = y_test - y_best_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Actual vs predicted (test set)
axes[0].scatter(y_test, y_best_pred, s=14, alpha=0.4, color='#4C78A8', edgecolors='none')
lims = [0, max(y_test.max(), y_best_pred.max())]
axes[0].plot(lims, lims, '--', color='#555555', lw=1.5)
axes[0].set_xlabel('Actual price (billion VND)')
axes[0].set_ylabel('Predicted price (billion VND)')
axes[0].set_title('Actual vs predicted — test set')
axes[0].grid(True, alpha=0.25)
axes[0].spines[['top', 'right']].set_visible(False)

# Error distribution
axes[1].hist(errors, bins=40, color='#4C78A8', edgecolor='white', linewidth=0.5)
axes[1].axvline(errors.mean(), color='#F58518', linestyle='--', lw=1.5,
                label=f'Mean error: {errors.mean():.2f}')
axes[1].set_xlabel('Prediction error (billion VND)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Error distribution — test set')
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.25)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 1.9 Feature importance

In [ ]:
# TabPFN không có feature_importances_ — nếu nó là best model thì xem importance của Random Forest
if hasattr(best_model, 'feature_importances_'):
    imp_model, imp_name = best_model, best_name
else:
    imp_model, imp_name = rf, 'random_forest'

importance = (
    pd.Series(imp_model.feature_importances_, index=FEATURE_COLS)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 9))
ax.barh(importance.index, importance.values, height=0.62, color='#4C78A8')
ax.set_xlabel('Importance')
ax.set_title(f'Feature importance — {imp_name.replace("_", " ")}')
ax.grid(True, axis='x', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 1.10 Lưu model

In [ ]:
model_dir = Path('saved_models')
model_dir.mkdir(exist_ok=True)

model_path = model_dir / f"{best_name}.joblib"
joblib.dump(best_model, model_path)
print(f"✓ Saved model: {model_path}")

metadata = {
    'model_type': best_name,
    'features': FEATURE_COLS,
    'metrics': {
        'r2_score': float(best_r2),
        'rmse': float(np.sqrt(mean_squared_error(y_test, y_best_pred))),
        'mae': float(mean_absolute_error(y_test, y_best_pred)),
    },
    'train_size': len(X_train),
    'test_size': len(X_test),
}
meta_path = model_dir / f"{best_name}_meta.pkl"
with open(meta_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✓ Saved metadata: {meta_path}")

## Part 2 — Inference

Load model đã lưu và dự đoán giá cho **toàn bộ** listings (dùng `df` đã load ở Part 1).

> ⚠️ Nếu model tốt nhất là TabPFN: predict 8.581 dòng trên CPU có thể mất **hàng chục giờ**.
> Trường hợp đó nên dùng Random Forest cho Part 2 (sửa cell 2.1: `latest_model = model_dir / 'random_forest.joblib'`).

### 2.1 Load model đã lưu

In [ ]:
model_dir = Path('saved_models')
model_files = sorted(model_dir.glob('*.joblib'))
assert model_files, "No trained models found — run Part 1 first"

latest_model = model_files[-1]
model = joblib.load(latest_model)

with open(model_dir / f"{latest_model.stem}_meta.pkl", 'rb') as f:
    metadata = pickle.load(f)

print(f"✓ Loaded {metadata['model_type']}")
print(f"  R²: {metadata['metrics']['r2_score']:.4f} | RMSE: {metadata['metrics']['rmse']:.4f} billion VND")

### 2.2 Dự đoán cho toàn bộ listings

In [ ]:
features = metadata['features']

# Build the feature matrix the same way as in training (build_features from Part 1)
X_all = build_features(df).reindex(columns=features, fill_value=0)

df['predicted_price_billion_vnd'] = model.predict(X_all)
print(f"✓ Predicted {len(df)} listings")
print(f"Prediction range: {df['predicted_price_billion_vnd'].min():.2f} – {df['predicted_price_billion_vnd'].max():.2f} billion VND")

df[['area_m2', 'distance_to_center_km', 'price_billion_vnd', 'predicted_price_billion_vnd']].head(10)

### 2.3 Đánh giá dự đoán

In [ ]:
df['error_billion_vnd'] = df['predicted_price_billion_vnd'] - df['price_billion_vnd']

mae_all = df['error_billion_vnd'].abs().mean()
rmse_all = np.sqrt((df['error_billion_vnd'] ** 2).mean())
print("Prediction error on all listings (with known price):")
print(f"  MAE:  {mae_all:.4f} billion VND")
print(f"  RMSE: {rmse_all:.4f} billion VND")
print(f"  Mean error: {df['error_billion_vnd'].mean():.4f} billion VND")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(df['price_billion_vnd'], df['predicted_price_billion_vnd'],
                s=12, alpha=0.35, color='#4C78A8', edgecolors='none')
lims = [0, df['price_billion_vnd'].quantile(0.99)]
axes[0].plot(lims, lims, '--', color='#555555', lw=1.5)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual price (billion VND)')
axes[0].set_ylabel('Predicted price (billion VND)')
axes[0].set_title('Actual vs predicted — all listings (99th pct zoom)')
axes[0].grid(True, alpha=0.25)
axes[0].spines[['top', 'right']].set_visible(False)

err = df['error_billion_vnd'].dropna()
axes[1].hist(err[err.abs() < err.abs().quantile(0.99)], bins=50,
             color='#4C78A8', edgecolor='white', linewidth=0.5)
axes[1].axvline(err.mean(), color='#F58518', linestyle='--', lw=1.5,
                label=f'Mean error: {err.mean():.2f}')
axes[1].set_xlabel('Prediction error (billion VND)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Error distribution — all listings')
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.25)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 2.4 Lưu kết quả dự đoán

In [ ]:
output_file = data_dir / 'predictions_latest.csv'
df.to_csv(output_file, index=False)

print(f"✓ Saved predictions: {output_file}")
print(f"  Records: {len(df)} | Size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
print("\n✅ Done!")